# Analise exploratoria: Adventure Works

Entregavel 1 do desafio: analise exploratoria e explanatoria, com codigo, graficos
e comentario em cada insight.

A analise roda sobre os marts gerados pelo dbt, nao sobre as tabelas cruas. Rode
`dbt build` antes deste notebook.

**Pergunta que guia a analise:** a area comercial investe em acoes promocionais
porque acredita que elas geram resultado. Os dados sustentam isso?


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

con = duckdb.connect("../adventureworks.duckdb", read_only=True)

def q(sql):
    return con.execute(sql).df()

q("select count(*) as itens, count(distinct sales_order_id) as pedidos from marts.fct_sales")

## 1. Volumetria e periodo coberto

Primeiro confirmamos o tamanho e a janela dos dados, para saber o que pode e o que
nao pode ser afirmado.

In [ ]:
q("""
select
    min(date_key)                     as primeira_venda,
    max(date_key)                     as ultima_venda,
    count(distinct sales_order_id)    as pedidos,
    count(*)                          as itens,
    count(distinct customer_key)      as clientes,
    count(distinct product_key)       as produtos,
    sum(net_revenue)                  as receita_liquida
from marts.fct_sales
""")

**Insight 1.** ESCREVA AQUI o que a volumetria revela: a janela real dos dados, se
ha ano incompleto na ponta (o que distorce comparacoes anuais) e a ordem de grandeza
da receita. Se o ultimo ano estiver parcial, toda analise YoY precisa ser feita em
periodo equivalente.

## 2. Qualidade dos dados

Antes de analisar, checamos se ha algo que invalide conclusoes: nulos em chave,
quantidades ou valores negativos, e a coerencia aritmetica das metricas.

In [ ]:
q("""
select
    sum(case when customer_key is null then 1 else 0 end)        as sem_cliente,
    sum(case when product_key is null then 1 else 0 end)         as sem_produto,
    sum(case when geography_key is null then 1 else 0 end)       as sem_geografia,
    sum(case when order_qty <= 0 then 1 else 0 end)              as qtd_invalida,
    sum(case when net_revenue < 0 then 1 else 0 end)             as receita_negativa,
    sum(case when abs((gross_revenue - discount_amount) - net_revenue) > 0.01
             then 1 else 0 end)                                  as metrica_incoerente
from marts.fct_sales
""")

**Insight 2.** ESCREVA AQUI. O esperado e tudo zerado, porque os mesmos criterios estao
cobertos por testes automatizados no dbt, entao uma falha aqui significaria que o
`dbt test` tambem falhou. Vale dizer isso na apresentacao: a qualidade nao e
verificada uma vez, e verificada a cada execucao.

## 3. Validacao contra a auditoria

O CEO pediu que os numeros batessem com a contabilidade: vendas brutas de 2011 =
12.646.112,16. Esta celula reproduz a conferencia que o teste do dbt faz.

In [ ]:
q("""
select
    extract(year from date_key) as ano,
    count(distinct sales_order_id) as pedidos,
    sum(gross_revenue) as receita_bruta,
    sum(discount_amount) as descontos,
    sum(net_revenue) as receita_liquida
from marts.fct_sales
group by 1
order by 1
""")

**Insight 3.** ESCREVA AQUI qual definicao de "vendas brutas" bate com os
12.646.112,16 informados pelo CEO. Registre a definicao em
`docs/regras-de-negocio.md`. E essa a conferencia que sustenta a confianca no
restante do painel.

## 4. Receita ao longo do tempo

Serie mensal de receita e pedidos. Responde a pergunta "e" do enunciado e revela
sazonalidade.

In [ ]:
serie = q("""
select
    d.year_month,
    count(distinct f.sales_order_id) as pedidos,
    sum(f.net_revenue) as receita
from marts.fct_sales f
join marts.dim_date d on f.date_key = d.date_key
group by 1
order by 1
""")

fig, ax = plt.subplots()
ax.plot(serie["year_month"], serie["receita"], linewidth=1.8, color="#2e3fb8")
ax.set_title("Receita liquida por mes")
ax.set_ylabel("receita")
ax.set_xticks(serie["year_month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()

serie.tail(12)

**Insight 4.** ESCREVA AQUI o padrao da serie: tendencia geral, picos recorrentes
(o AdventureWorks concentra vendas em fim de trimestre) e qualquer degrau abrupto.
Um degrau costuma indicar mudanca de operacao, nao de demanda. Vale checar se
coincide com a entrada do canal online.

## 5. Canal de venda

`is_online_order` separa venda direta ao consumidor de venda para revenda. Sao
negocios com ticket e comportamento diferentes, e misturar os dois esconde ambos.

In [ ]:
canal = q("""
select
    case when is_online_order then 'Online' else 'Revenda' end as canal,
    count(distinct sales_order_id) as pedidos,
    sum(net_revenue) as receita,
    sum(net_revenue) / count(distinct sales_order_id) as ticket_medio
from marts.fct_sales
group by 1
""")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(canal["canal"], canal["receita"], color="#2e3fb8", width=.55)
ax[0].set_title("Receita por canal")
ax[1].bar(canal["canal"], canal["ticket_medio"], color="#9fa3e0", width=.55)
ax[1].set_title("Ticket medio por canal")
for a in ax:
    a.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()

canal

**Insight 5.** ESCREVA AQUI a diferenca entre os canais. No AdventureWorks a
revenda costuma responder pela maior parte da receita com um numero pequeno de
pedidos de valor alto, enquanto o online tem muitos pedidos pequenos. Se for o caso,
a consequencia pratica e que qualquer meta unica de ticket medio e enganosa.

## 6. Concentracao de clientes

Quanto da receita depende dos maiores compradores. E um risco comercial que
normalmente ninguem acompanha.

In [ ]:
top_clientes = q("""
select
    c.customer_name,
    c.customer_type,
    sum(f.net_revenue) as receita
from marts.fct_sales f
join marts.dim_customer c on f.customer_key = c.customer_key
group by 1, 2
order by receita desc
limit 10
""")

total = q("select sum(net_revenue) as t from marts.fct_sales")["t"][0]
participacao = top_clientes["receita"].sum() / total

fig, ax = plt.subplots()
ax.barh(top_clientes["customer_name"][::-1], top_clientes["receita"][::-1], color="#2e3fb8")
ax.set_title(f"Top 10 clientes: {participacao:.1%} da receita total")
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.show()

print(f"Os 10 maiores clientes respondem por {participacao:.1%} da receita.")
top_clientes

**Insight 6.** ESCREVA AQUI com o numero que saiu. Se a concentracao for alta,
e uma recomendacao acionavel direta: a perda de um unico cliente tem impacto
material e hoje nao existe alerta para isso.

## 7. Mix de produto

In [ ]:
mix = q("""
select
    p.category_name,
    sum(f.net_revenue) as receita,
    sum(f.order_qty) as unidades
from marts.fct_sales f
join marts.dim_product p on f.product_key = p.product_key
group by 1
order by receita desc
""")

fig, ax = plt.subplots()
ax.bar(mix["category_name"], mix["receita"], color="#e49ed4", width=.6)
ax.set_title("Receita por categoria de produto")
ax.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()

mix

**Insight 7.** ESCREVA AQUI. Compare receita com unidades: categorias com muitas
unidades e pouca receita sao acessorios de baixo valor, e o oposto indica
dependencia de poucos produtos caros.

## 8. A pergunta que interessa a Diretoria Comercial

A tese da area e que acoes promocionais geram resultado. Aqui comparamos pedidos
com e sem o motivo "Promotion" em taxa de desconto e ticket medio.

Se a promocao entrega muito desconto sem elevar o ticket, ela esta transferindo
margem, nao criando demanda.

In [ ]:
promo = q("""
select
    case when is_promotion then 'Com promocao' else 'Sem promocao' end as grupo,
    count(distinct sales_order_id) as pedidos,
    sum(order_qty) as unidades,
    sum(gross_revenue) as receita_bruta,
    sum(discount_amount) as descontos,
    sum(discount_amount) / nullif(sum(gross_revenue), 0) as taxa_desconto,
    (sum(gross_revenue) - sum(discount_amount)) / count(distinct sales_order_id) as ticket_medio
from marts.fct_sales
group by 1
""")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(promo["grupo"], promo["taxa_desconto"], color=["#9a5b00", "#2e3fb8"], width=.55)
ax[0].set_title("Taxa de desconto")
ax[1].bar(promo["grupo"], promo["ticket_medio"], color=["#9a5b00", "#2e3fb8"], width=.55)
ax[1].set_title("Ticket medio")
for a in ax:
    a.grid(axis="y", alpha=.25)
plt.tight_layout()
plt.show()

promo

**Insight 8, o principal.** ESCREVA AQUI a conclusao com os numeros na mao.
Este e o insight que sustenta a argumentacao da apresentacao, porque responde com
dado a uma crenca que hoje e defendida por opiniao. Seja honesto com o resultado:
se a promocao estiver funcionando, isso tambem e um achado util, e mostra que o
painel serve para confirmar decisoes, nao so para contestar.

## 9. Produto mais vendido em promocao

Responde a pergunta "f" do enunciado.

In [ ]:
q("""
select
    p.product_name,
    sum(f.order_qty) as unidades
from marts.fct_sales f
join marts.dim_product p on f.product_key = p.product_key
where f.is_promotion
group by 1
order by unidades desc
limit 10
""")

## 10. Top 5 cidades

Responde a pergunta "d" do enunciado.

In [ ]:
q("""
select
    g.city,
    g.state_province,
    g.country,
    sum(f.net_revenue) as receita
from marts.fct_sales f
join marts.dim_geography g on f.geography_key = g.geography_key
group by 1, 2, 3
order by receita desc
limit 5
""")

## Conclusoes

Resuma aqui, em quatro ou cinco linhas, os achados que vao para o dashboard e para
a apresentacao. Cada um deve virar um visual ou uma recomendacao. Insight que nao
muda decisao nao precisa estar no painel.

In [ ]:
con.close()